In [117]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import  torch.optim as optim
from sklearn.metrics import r2_score

# Load the DataSet

In [7]:
df_ori=pd.read_csv("powerplant_data.csv")
df=df_ori.copy()

# Serparte the x & y 

In [17]:
x=df.drop(columns=["PE"])
y=df["PE"]

# Split the data for Training & testing

In [23]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# Standarize the Data

In [30]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# Step-1 we need to convet our data to DataLoader . 
 -> convert the data into tensor. 
 ->then convert the tensor's to tensor Dataset
 ->then convert tensorDataset into Dataloader

# Convert the data into Tensor's 

In [55]:
# convert training data into tensor's 
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

# convert testing data into tensor's 
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

# Convert the data into TensorDataset

In [77]:
# here what we hvae done is we have taken the tensor of x_train,x_test && y_train,y_test 
# and we have made 2 set's of tensorDataset
train_tensor_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_tensor_dataset=TensorDataset(x_test_tensor,y_test_tensor)

# Convert the TensorDataset into DataLoader

In [78]:
training_data_loader=DataLoader(train_tensor_dataset,shuffle=True,batch_size=32)
test_data_loader=DataLoader(test_tensor_dataset,batch_size=32)

# Step-2 Data is Ready . Now we Create create the ANN 

# Createing ANN

In [79]:
class ANN(nn.Module):
    # create the constructor
    def __init__(self):
        super(ANN,self).__init__()
        # create the nn  aircheture
        self.model=nn.Sequential(
            # 1st layer
            nn.Linear(x_train.shape[1],6), #here the input is the shape of input features. which is 4 
            nn.ReLU(),    #here we are applying the relu activation fxn

            #2nd layer
            nn.Linear(6,6),
            nn.ReLU(),

            #output layer will be 6,1
            nn.Linear(6,1),
        )
    # create the forwared fxn
    def forward(self,x):
        return self.model(x)
        

# Step-3 Train the ANN

In [84]:
model=ANN()   #init the model 
createrion=nn.MSELoss()    #set the loss Fxn
optimizer=optim.Adam(model.parameters())    #define the optimzer

In [ ]:

# define the 
epochs=100
overall_training_loss=[]
overall_testing_loss=[]
best_val_loss=float("inf")

for epoc in range(epochs):
    # put the model in training mode so that it update it's weight's 
    model.train()
    training_loss=0.0
    # train all the batches one by one that are in dataLoader
    for xb,yb in training_data_loader:
        # make the gradients values to zero again that were computed 
        optimizer.zero_grad()             # in the last iteration
        # train the model by doing forwared propagation
        output=model(xb)
        # calculate the loss 
        loss=createrion(output,yb)
        # do the backwared propageation and compute the gradient's
        loss.backward()
        # using the ADAM optimizer to udpate the weight with step fxn
        optimizer.step()
        #add the losss to the training lostt
        training_loss+=loss.item()  #here loss is a tenser convet =>float
    epoc_train_loss=training_loss/len(training_data_loader)
    print(f"training loss after {epoc}th epoc is      ",epoc_train_loss)
    # add the loss to the voerall array
    overall_training_loss.append(epoc_train_loss)

    # now test the model on the each batch
    validation_loss=0.0
    model.eval()
    with torch.no_grad():  #no computation of gradeients
        for xb,yb in test_data_loader:
            output=model(xb)
            loss=createrion(output,yb)
            validation_loss+=loss.item() #add the valdation loss
            
    epoc_val_loss=validation_loss/len(test_data_loader)
    print(f"validation loss after {epoc}th epoc is      ",epoc_val_loss)
    overall_testing_loss.append(epoc_val_loss)

    if epoc_val_loss <best_val_loss:
        best_val_loss=epoc_val_loss

# Save the Best Model

In [88]:
torch.save(model.state_dict(),"best_model.pt")

# Load the Saved Model and test it on the testing data 

# Evaluation

In [114]:
model.load_state_dict(torch.load("best_model.pt"))

createrian=nn.MSELoss()

In [120]:
model.eval()
with torch.no_grad():
    test_pred=model(x_test_tensor)
    loss=createrian(test_pred,y_test_tensor)
    r2 =r2_score(y_test,test_pred)
    print(f"the r2 score is ",r2)

the r2 score is  0.929018966150822


function